# 05 — FFN as Memory: What Do MLP Layers Store?

The MLP (feed-forward) layers contain ~2/3 of all parameters. Research suggests they function as **key-value memories**: the gate/up projections act as "keys" that match input patterns, and the down projection stores "values" that get written into the residual stream.

This notebook explores the SwiGLU MLP step by step and looks for interpretable neurons.

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, "..")

import torch
import numpy as np
import matplotlib.pyplot as plt
from utils.model_loading import load_tlens_model, format_param_count
from utils.visualization import (
    apply_theme, ACCENT_BLUE, ACCENT_ORANGE, ACCENT_GREEN, ACCENT_RED,
    ACCENT_PURPLE, ACCENT_TEAL, TEXT_COLOR, DARK_BG, DARK_SURFACE, DARK_GRID,
    PALETTE, _style_box,
)
apply_theme()

MODEL_SIZE = "0.5b"
model = load_tlens_model(MODEL_SIZE)
cfg = model.cfg

print(f"MLP per layer: {cfg.d_model} -> {cfg.d_mlp} -> {cfg.d_model}")
print(f"Expansion ratio: {cfg.d_mlp / cfg.d_model:.1f}x")
print(f"MLP params per layer: {format_param_count(3 * cfg.d_model * cfg.d_mlp)}")

## SwiGLU Step by Step

Let's trace what happens inside the MLP for a single token.

In [ ]:
prompt = "The capital of France is"
logits, cache = model.run_with_cache(prompt)
tokens = model.to_str_tokens(prompt)

# Get the hidden state going into layer 12's MLP (after attention + norm)
layer = 12
resid_mid = cache[f"blocks.{layer}.hook_resid_mid"][0]  # after attention, before MLP
ln2_out = model.blocks[layer].ln2(resid_mid)  # after pre-MLP norm

# Get the last token's hidden state
x = ln2_out[-1]  # shape: [d_model]

# Step through SwiGLU manually
W_gate = model.blocks[layer].mlp.W_gate  # [d_model, d_mlp]
W_in = model.blocks[layer].mlp.W_in      # [d_model, d_mlp]
W_out = model.blocks[layer].mlp.W_out     # [d_mlp, d_model]

gate_pre = x @ W_gate          # [d_mlp] — pre-activation gate
gate_post = torch.nn.functional.silu(gate_pre)  # SiLU activation
up = x @ W_in                  # [d_mlp] — up projection
gated = gate_post * up         # element-wise gating
mlp_out = gated @ W_out        # [d_model] — output

# Visualize each step
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(gate_pre.detach().float().cpu().numpy(), bins=100, color=ACCENT_GREEN, alpha=0.8, edgecolor="none")
axes[0, 0].set_title(f"Gate Pre-Activation (x @ W_gate)")
axes[0, 0].axvline(0, color=TEXT_COLOR, linestyle="--", alpha=0.3)

axes[0, 1].hist(gate_post.detach().float().cpu().numpy(), bins=100, color=ACCENT_GREEN, alpha=0.8, edgecolor="none")
axes[0, 1].set_title(f"Gate Post-Activation (SiLU)")
axes[0, 1].axvline(0, color=TEXT_COLOR, linestyle="--", alpha=0.3)

axes[1, 0].hist(gated.detach().float().cpu().numpy(), bins=100, color=ACCENT_GREEN, alpha=0.8, edgecolor="none")
axes[1, 0].set_title(f"After Gating (gate * up)")
axes[1, 0].axvline(0, color=TEXT_COLOR, linestyle="--", alpha=0.3)

# Show sparsity: how many neurons are effectively "off"
gated_abs = gated.detach().float().cpu().abs()
threshold = gated_abs.max() * 0.01
n_active = (gated_abs > threshold).sum().item()
n_total = gated_abs.shape[0]

axes[1, 1].bar(["Active", "Inactive"], [n_active, n_total - n_active],
               color=[ACCENT_GREEN, ACCENT_RED])
axes[1, 1].set_title(f"Neuron Activity: {n_active}/{n_total} active ({100*n_active/n_total:.1f}%)")

plt.tight_layout()
plt.show()
print(f"The SwiGLU gate effectively silences {100*(1-n_active/n_total):.0f}% of neurons.")

## Gate Sparsity Across Layers

How much of the MLP is "active" at each layer? Sparser activation means the model is more selective about which neurons it uses.

In [ ]:
# Analyze gate sparsity at every layer for the last token
sparsity_per_layer = []

for layer_idx in range(cfg.n_layers):
    # Get MLP intermediate activations (post-gating)
    mlp_post = cache[f"blocks.{layer_idx}.mlp.hook_post"][0, -1]  # last token
    abs_vals = mlp_post.detach().float().cpu().abs()
    threshold = abs_vals.max() * 0.01
    active_frac = (abs_vals > threshold).float().mean().item()
    sparsity_per_layer.append(active_frac)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(cfg.n_layers), [s * 100 for s in sparsity_per_layer],
       color=ACCENT_GREEN, alpha=0.8)
ax.set_xlabel("Layer")
ax.set_ylabel("Active Neurons (%)")
ax.set_title("MLP Gate Sparsity by Layer (last token of prompt)")
ax.axhline(np.mean(sparsity_per_layer) * 100, color=ACCENT_ORANGE, linestyle="--",
           label=f"Mean: {np.mean(sparsity_per_layer)*100:.1f}%")
ax.legend()
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

## Finding Interpretable Neurons

Let's run several diverse prompts and find neurons that activate selectively for specific content types.

In [ ]:
# Run diverse prompts and collect MLP activations at a mid layer
test_prompts = [
    "The price is 42 dollars and 99 cents",        # numbers
    "def fibonacci(n):\n    return",                # code
    "Paris, London, Tokyo, and Berlin are",         # cities/geography
    "She was extremely happy and grateful",          # emotion
    "The chemical formula for water is H2O",         # science
    "Yesterday morning, I went to the store",        # narrative
]

layer_to_inspect = 12
all_activations = {}  # neuron_idx -> {prompt_idx: activation_value}

for prompt_idx, p in enumerate(test_prompts):
    _, p_cache = model.run_with_cache(p)
    # Get last token's MLP activations
    acts = p_cache[f"blocks.{layer_to_inspect}.mlp.hook_post"][0, -1].detach().float().cpu()
    for neuron_idx in range(cfg.d_mlp):
        if neuron_idx not in all_activations:
            all_activations[neuron_idx] = []
        all_activations[neuron_idx].append(acts[neuron_idx].item())

# Find neurons with highest variance across prompts (= most selective)
variances = {n: np.var(acts) for n, acts in all_activations.items()}
top_selective = sorted(variances, key=variances.get, reverse=True)[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
short_labels = [p[:30] + "..." if len(p) > 30 else p for p in test_prompts]

for idx, neuron_idx in enumerate(top_selective):
    acts = all_activations[neuron_idx]
    colors = [ACCENT_GREEN if a > 0 else ACCENT_RED for a in acts]
    axes[idx].barh(range(len(acts)), acts, color=colors)
    axes[idx].set_yticks(range(len(acts)))
    axes[idx].set_yticklabels(short_labels, fontsize=7)
    axes[idx].set_title(f"Neuron {neuron_idx} (var={variances[neuron_idx]:.2f})")
    axes[idx].axvline(0, color=TEXT_COLOR, linewidth=0.5)

plt.suptitle(f"Most Selective Neurons in Layer {layer_to_inspect}", fontsize=14)
plt.tight_layout()
plt.show()
print("Selective neurons fire strongly for some prompts and stay quiet for others.")

## The Key-Value Memory Interpretation

Each MLP neuron can be viewed as a key-value pair:
- **Key** (row of W_gate/W_in): the input pattern the neuron responds to
- **Value** (column of W_out): what the neuron writes to the residual stream when activated

The neuron fires when the input matches its key, and its value gets added to the residual. This is how MLP layers store and retrieve factual associations.

In [ ]:
# For the most active neuron on our "France" prompt, show its "value" vector
# (what it writes to the residual stream) by projecting through the unembedding

most_active_neuron = top_selective[0]
value_vector = W_out[most_active_neuron]  # column of W_out, shape: [d_model]

# What tokens does this value promote?
normed_value = model.ln_final(value_vector.unsqueeze(0))
value_logits = model.unembed(normed_value)[0]
value_probs = torch.softmax(value_logits, dim=-1)

top_promoted = torch.topk(value_probs, 15)
print(f"Neuron {most_active_neuron}'s 'value' — top tokens it promotes when active:")
for prob, idx in zip(top_promoted.values, top_promoted.indices):
    print(f"  {prob:.4f}  '{model.to_string(idx.item())}'")

print(f"\nThis neuron's activation on the France prompt: {all_activations[most_active_neuron][-1]:.3f}")
print("When this neuron fires, it pushes the output distribution toward these tokens.")